# nb_02_silver_transform
**Milestone 2: Silver layer.** Reads the 5 `bronze_` tables, converts text to proper data types, standardises values, runs data quality (DQ) rules, quarantines bad rows, and writes 5 clean `silver_` tables.

**Outputs:** `silver_customers`, `silver_products`, `silver_stores`, `silver_exchange_rates`, `silver_sales`, `silver_dq_rejects` (quarantined rows), `silver_dq_results` (rule results per run).

### Cell 1: Parameters
A pipeline can override this. If more than this % of any table is rejected, the run fails.

In [ ]:
max_reject_pct = 1.0   # fail the run if more than 1% of any table is rejected

### Cell 2: Helper functions
- `try_to_timestamp` / `try_cast` return **NULL** for bad values instead of crashing the whole job. A DQ rule then catches the NULL and quarantines the row.
- `dup_flag` marks the 2nd+ copy of a business key, keeping the latest loaded.

In [ ]:
import uuid
from pyspark.sql import functions as F, Window

batch_id = str(uuid.uuid4())
DATE_FMT = "M/d/yyyy"

def to_date_safe(col_name):
    """Text 'M/d/yyyy' -> date. Bad or blank values become NULL instead of crashing the job."""
    return F.to_date(F.try_to_timestamp(F.col(col_name), F.lit(DATE_FMT)))

def to_int_safe(col_name):
    return F.expr(f"try_cast(trim({col_name}) AS INT)")

def to_money_safe(col_name):
    """'$2,899.99 ' -> 2899.99 as DECIMAL(10,2)"""
    return F.expr(f"try_cast(regexp_replace({col_name}, '[$, ]', '') AS DECIMAL(10,2))")

def clean_text(col_name):
    """Trim spaces; turn empty strings into NULL."""
    c = F.trim(F.col(col_name))
    return F.when(c == "", None).otherwise(c)

def dup_flag(df, key_cols):
    """True for 2nd, 3rd... copies of the same business key."""
    w = Window.partitionBy(*key_cols).orderBy(F.col("_ingested_at").desc())
    return F.row_number().over(w) > 1

### Cell 3: The DQ engine
- **error** rules: the row goes to `silver_dq_rejects`, with the rule names and the original raw text.
- **warning** rules: the row is kept, and the count is logged.
- Every rule's count goes to `silver_dq_results`, so you have DQ history across runs.

In [ ]:
dq_results = []   # one row per table per rule -> silver_dq_results

def apply_dq(df, table_name, key_cols, rules):
    """
    rules: list of (rule_name, severity, bad_condition)
      severity 'error'   -> row is quarantined to silver_dq_rejects
      severity 'warning' -> row is kept, but the count is logged
    Returns only the good rows.
    """
    total = df.count()
    for name, severity, cond in rules:
        df = df.withColumn(f"_dq_{name}", F.coalesce(cond, F.lit(False)))
    # one pass to count failures for every rule
    counts = df.agg(*[F.sum(F.col(f"_dq_{n}").cast("int")).alias(n) for n, _, _ in rules]).first()
    for name, severity, _ in rules:
        dq_results.append((batch_id, table_name, name, severity, int(counts[name] or 0), total))

    error_rules = [n for n, s, _ in rules if s == "error"]
    failed_list = F.array_compact(F.array(*[F.when(F.col(f"_dq_{n}"), F.lit(n)) for n in error_rules])) \
                  if error_rules else F.array().cast("array<string>")
    df = df.withColumn("_failed_rules", failed_list)

    rejects = (df.filter(F.size("_failed_rules") > 0)
                 .select(F.lit(table_name).alias("table_name"),
                         F.concat_ws("|", *[F.col(k).cast("string") for k in key_cols]).alias("record_key"),
                         F.array_join("_failed_rules", ", ").alias("failed_rules"),
                         F.to_json("_raw").alias("raw_record"),
                         F.lit(batch_id).alias("_batch_id"),
                         F.current_timestamp().alias("_rejected_at")))
    (rejects.write.format("delta").mode("append").saveAsTable("silver_dq_rejects"))

    good = (df.filter(F.size("_failed_rules") == 0)
              .drop("_raw", "_failed_rules", *[f"_dq_{n}" for n, _, _ in rules]))
    return good

def write_silver(df, table_name):
    (df.withColumn("_batch_id", F.lit(batch_id))
       .withColumn("_processed_at", F.current_timestamp())
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(table_name))
    print(f"{table_name:<24} {spark.table(table_name).count():>7,} rows written")

### Cell 4: Customers
Title-cases the 25% of cities stored in ALL CAPS (only those; mixed-case names are left alone). Zip stays text because of Canadian postcodes. Birthday text becomes a real date.

In [ ]:
b = spark.table("bronze_customers")
customers = b.select(
    to_int_safe("customer_key").alias("customer_key"),
    clean_text("gender").alias("gender"),
    clean_text("name").alias("customer_name"),
    # 25% of cities are ALL CAPS -> Title Case; mixed-case cities are left as they are
    F.when(F.trim("city") == F.upper(F.trim("city")), F.initcap(F.trim("city")))
     .otherwise(F.trim("city")).alias("city"),
    clean_text("state_code").alias("state_code"),
    clean_text("state").alias("state"),
    clean_text("zip_code").alias("zip_code"),          # stays text: Canadian postcodes are letters+digits
    clean_text("country").alias("country"),
    clean_text("continent").alias("continent"),
    to_date_safe("birthday").alias("birth_date"),
    F.col("_ingested_at"),
    F.struct(*[F.col(c) for c in b.columns if not c.startswith("_")]).alias("_raw"),
)
customers = apply_dq(customers, "customers", ["customer_key"], [
    ("customer_key_invalid",  "error",   F.col("customer_key").isNull()),
    ("customer_key_duplicate","error",   dup_flag(customers, ["customer_key"])),
    ("birth_date_invalid",    "error",   F.col("birth_date").isNull()),
    ("birth_date_implausible","warning", (F.col("birth_date") > F.current_date()) |
                                         (F.col("birth_date") < F.lit("1900-01-01").cast("date"))),
    ("gender_unexpected",     "warning", ~F.col("gender").isin("Male", "Female")),
]).drop("_ingested_at")
write_silver(customers, "silver_customers")

### Cell 5: Products
`'$2,899.99 '` becomes `2899.99` as DECIMAL(10,2). DECIMAL, not float, so money adds up exactly. Category keys stay text to keep the leading zero (`0101`).

In [ ]:
b = spark.table("bronze_products")
products = b.select(
    to_int_safe("product_key").alias("product_key"),
    clean_text("product_name").alias("product_name"),
    clean_text("brand").alias("brand"),
    clean_text("color").alias("color"),
    to_money_safe("unit_cost_usd").alias("unit_cost_usd"),
    to_money_safe("unit_price_usd").alias("unit_price_usd"),
    clean_text("subcategory_key").alias("subcategory_key"),   # text: keeps leading zero '0101'
    clean_text("subcategory").alias("subcategory"),
    clean_text("category_key").alias("category_key"),
    clean_text("category").alias("category"),
    F.col("_ingested_at"),
    F.struct(*[F.col(c) for c in b.columns if not c.startswith("_")]).alias("_raw"),
)
products = apply_dq(products, "products", ["product_key"], [
    ("product_key_invalid",   "error",   F.col("product_key").isNull()),
    ("product_key_duplicate", "error",   dup_flag(products, ["product_key"])),
    ("price_invalid",         "error",   F.col("unit_price_usd").isNull() | (F.col("unit_price_usd") <= 0)),
    ("cost_invalid",          "error",   F.col("unit_cost_usd").isNull() | (F.col("unit_cost_usd") <= 0)),
    ("price_below_cost",      "warning", F.col("unit_price_usd") < F.col("unit_cost_usd")),
]).drop("_ingested_at")
write_silver(products, "silver_products")

### Cell 6: Stores
Store 0 is the Online store, so it has no floor size. That's flagged as `store_type = 'Online'`, not treated as an error.

In [ ]:
b = spark.table("bronze_stores")
stores = b.select(
    to_int_safe("store_key").alias("store_key"),
    clean_text("country").alias("country"),
    clean_text("state").alias("state"),
    to_int_safe("square_meters").alias("square_meters"),   # NULL for the Online store
    to_date_safe("open_date").alias("open_date"),
    F.col("_ingested_at"),
    F.struct(*[F.col(c) for c in b.columns if not c.startswith("_")]).alias("_raw"),
).withColumn("store_type", F.when(F.col("store_key") == 0, "Online").otherwise("Physical"))
stores = apply_dq(stores, "stores", ["store_key"], [
    ("store_key_invalid",        "error",   F.col("store_key").isNull()),
    ("store_key_duplicate",      "error",   dup_flag(stores, ["store_key"])),
    ("open_date_invalid",        "error",   F.col("open_date").isNull()),
    ("physical_store_no_size",   "warning", (F.col("store_type") == "Physical") & F.col("square_meters").isNull()),
]).drop("_ingested_at")
write_silver(stores, "silver_stores")

### Cell 7: Exchange rates
The rate means **local currency per 1 USD** (USD = 1). A warning fires if USD is ever not 1.

In [ ]:
b = spark.table("bronze_exchange_rates")
fx = b.select(
    to_date_safe("date").alias("rate_date"),
    F.upper(clean_text("currency")).alias("currency_code"),
    F.expr("try_cast(trim(exchange) AS DECIMAL(18,6))").alias("exchange_rate"),  # local currency per 1 USD
    F.col("_ingested_at"),
    F.struct(*[F.col(c) for c in b.columns if not c.startswith("_")]).alias("_raw"),
)
fx = apply_dq(fx, "exchange_rates", ["rate_date", "currency_code"], [
    ("rate_date_invalid",   "error", F.col("rate_date").isNull()),
    ("currency_invalid",    "error", F.col("currency_code").isNull() | (F.length("currency_code") != 3)),
    ("rate_invalid",        "error", F.col("exchange_rate").isNull() | (F.col("exchange_rate") <= 0)),
    ("rate_duplicate",      "error", dup_flag(fx, ["rate_date", "currency_code"])),
    ("usd_rate_not_one",    "warning", (F.col("currency_code") == "USD") & (F.col("exchange_rate") != 1)),
]).drop("_ingested_at")
write_silver(fx, "silver_exchange_rates")

### Cell 8: Sales
Runs **last**, because every sale is checked against the clean Silver dimensions (customer, product, store, and an FX rate for that date and currency).

**Business rule:** a blank delivery date is **expected** for in-store sales. Only Online orders are delivered. So it's a warning check, not an error.

In [ ]:
b = spark.table("bronze_sales")
sales = b.select(
    to_int_safe("order_number").alias("order_number"),
    to_int_safe("line_item").alias("line_item"),
    to_date_safe("order_date").alias("order_date"),
    to_date_safe("delivery_date").alias("delivery_date"),   # blank for in-store sales = expected
    to_int_safe("customer_key").alias("customer_key"),
    to_int_safe("store_key").alias("store_key"),
    to_int_safe("product_key").alias("product_key"),
    to_int_safe("quantity").alias("quantity"),
    F.upper(clean_text("currency_code")).alias("currency_code"),
    F.col("_ingested_at"),
    F.struct(*[F.col(c) for c in b.columns if not c.startswith("_")]).alias("_raw"),
)
sales = (sales
    .withColumn("is_online", F.col("store_key") == 0)
    .withColumn("delivery_days", F.datediff("delivery_date", "order_date"))
    # referential integrity: does each key exist in the clean Silver dimension?
    .join(spark.table("silver_customers").select("customer_key", F.lit(True).alias("_cust_ok")), "customer_key", "left")
    .join(spark.table("silver_products").select("product_key", F.lit(True).alias("_prod_ok")), "product_key", "left")
    .join(spark.table("silver_stores").select("store_key", F.lit(True).alias("_store_ok")), "store_key", "left")
    .join(spark.table("silver_exchange_rates").select(F.col("rate_date").alias("order_date"), "currency_code",
                                                      F.lit(True).alias("_fx_ok")),
          ["order_date", "currency_code"], "left"))
sales = apply_dq(sales, "sales", ["order_number", "line_item"], [
    ("order_key_invalid",       "error", F.col("order_number").isNull() | F.col("line_item").isNull()),
    ("order_line_duplicate",    "error", dup_flag(sales, ["order_number", "line_item"])),
    ("order_date_invalid",      "error", F.col("order_date").isNull()),
    ("quantity_invalid",        "error", F.col("quantity").isNull() | (F.col("quantity") <= 0)),
    ("customer_not_found",      "error", F.col("_cust_ok").isNull()),
    ("product_not_found",       "error", F.col("_prod_ok").isNull()),
    ("store_not_found",         "error", F.col("_store_ok").isNull()),
    ("fx_rate_not_found",       "error", F.col("_fx_ok").isNull()),
    ("delivery_before_order",   "error", F.col("delivery_days") < 0),
    ("online_missing_delivery", "warning", F.col("is_online") & F.col("delivery_date").isNull()),
    ("instore_has_delivery",    "warning", ~F.col("is_online") & F.col("delivery_date").isNotNull()),
]).drop("_ingested_at", "_cust_ok", "_prod_ok", "_store_ok", "_fx_ok")

sales = sales.select("order_number", "line_item", "order_date", "delivery_date", "delivery_days",
                     "customer_key", "store_key", "product_key", "quantity", "currency_code", "is_online")
write_silver(sales, "silver_sales")

### Cell 9: DQ summary and gate
Shows every rule's result, then fails the run if any table's reject rate is above `max_reject_pct`. Rejects are saved **before** failing, so you can investigate them.

In [ ]:
results_df = (spark.createDataFrame(dq_results,
                 "batch_id string, table_name string, rule_name string, severity string, failed_rows long, total_rows long")
              .withColumn("failed_pct", F.round(F.col("failed_rows") / F.col("total_rows") * 100, 3))
              .withColumn("run_at", F.current_timestamp()))
results_df.write.format("delta").mode("append").saveAsTable("silver_dq_results")
display(results_df.select("table_name", "rule_name", "severity", "failed_rows", "total_rows", "failed_pct"))

rejected = (results_df.filter("severity = 'error'")
            .groupBy("table_name").agg(F.max("total_rows").alias("total"))
            .join(spark.table("silver_dq_rejects").filter(F.col("_batch_id") == batch_id)
                  .groupBy("table_name").count(), "table_name", "left")
            .fillna(0, ["count"])
            .withColumn("reject_pct", F.round(F.col("count") / F.col("total") * 100, 3)))
display(rejected)

breaches = [r.table_name for r in rejected.collect() if r.reject_pct > max_reject_pct]
if breaches:
    raise Exception(f"DQ gate FAILED: reject rate above {max_reject_pct}% in {breaches}. See silver_dq_rejects.")
print(f"Silver complete. Batch {batch_id}")

---
## Validation
### Test 1: Prove the DQ rules work
The real data is clean, so nothing was rejected. That doesn't prove the rules work. This cell feeds in 6 fake rows (1 good, 5 broken) and checks that only the good one passes.

**Expected:** `Rows that passed: [900001]`, plus 5 rejects with reasons: order_date_invalid, quantity_invalid, customer_not_found, delivery_before_order, fx_rate_not_found.

In [ ]:
# ===== TEST CELL: prove the rules catch bad data =====
test_rows = [
    # order, line, order_date,  delivery,     cust,      store, product, qty,  currency
    ("900001", "1", "1/5/2019",  "",          "301",     "1",   "1",     "2",  "AUD"),   # good row
    ("900002", "1", "13/45/2019","",          "301",     "1",   "1",     "2",  "AUD"),   # bad date
    ("900003", "1", "1/5/2019",  "",          "301",     "1",   "1",     "-3", "AUD"),   # negative quantity
    ("900004", "1", "1/5/2019",  "",          "9999999", "1",   "1",     "1",  "AUD"),   # unknown customer
    ("900005", "1", "1/5/2019",  "1/1/2019",  "301",     "0",   "1",     "1",  "USD"),   # delivered before ordered
    ("900006", "1", "1/5/2019",  "",          "301",     "1",   "1",     "1",  "XYZ"),   # no FX rate for currency
]
cols = ["order_number","line_item","order_date","delivery_date","customer_key","store_key","product_key","quantity","currency_code"]
t = spark.createDataFrame(test_rows, cols).withColumn("_ingested_at", F.current_timestamp())

t = t.select(
    to_int_safe("order_number").alias("order_number"), to_int_safe("line_item").alias("line_item"),
    to_date_safe("order_date").alias("order_date"), to_date_safe("delivery_date").alias("delivery_date"),
    to_int_safe("customer_key").alias("customer_key"), to_int_safe("store_key").alias("store_key"),
    to_int_safe("product_key").alias("product_key"), to_int_safe("quantity").alias("quantity"),
    F.upper(clean_text("currency_code")).alias("currency_code"), F.col("_ingested_at"),
    F.struct(*[F.col(c) for c in cols]).alias("_raw"),
).withColumn("delivery_days", F.datediff("delivery_date", "order_date")) \
 .join(spark.table("silver_customers").select("customer_key", F.lit(True).alias("_cust_ok")), "customer_key", "left") \
 .join(spark.table("silver_exchange_rates").select(F.col("rate_date").alias("order_date"), "currency_code",
                                                   F.lit(True).alias("_fx_ok")), ["order_date","currency_code"], "left")

good = apply_dq(t, "TEST_sales", ["order_number", "line_item"], [
    ("order_date_invalid",    "error", F.col("order_date").isNull()),
    ("quantity_invalid",      "error", F.col("quantity").isNull() | (F.col("quantity") <= 0)),
    ("customer_not_found",    "error", F.col("_cust_ok").isNull()),
    ("fx_rate_not_found",     "error", F.col("_fx_ok").isNull()),
    ("delivery_before_order", "error", F.col("delivery_days") < 0),
])
print("Rows that passed:", [r.order_number for r in good.collect()])      # expect [900001]
display(spark.table("silver_dq_rejects").filter("table_name = 'TEST_sales'")
        .select("record_key", "failed_rules", "raw_record"))

### Test 1 cleanup
Remove the fake rows, so the rejects table only holds real data.

In [ ]:
spark.sql("DELETE FROM silver_dq_rejects WHERE table_name = 'TEST_sales'")
print("Test rows removed")

### Test 2: Row counts, Bronze vs Silver
Expected: identical counts, because nothing in the real data was rejected.

In [ ]:
%%sql
SELECT 'sales' AS tbl, (SELECT COUNT(*) FROM bronze_sales) AS bronze, (SELECT COUNT(*) FROM silver_sales) AS silver
UNION ALL SELECT 'customers', (SELECT COUNT(*) FROM bronze_customers), (SELECT COUNT(*) FROM silver_customers)
UNION ALL SELECT 'products',  (SELECT COUNT(*) FROM bronze_products),  (SELECT COUNT(*) FROM silver_products)
UNION ALL SELECT 'stores',    (SELECT COUNT(*) FROM bronze_stores),    (SELECT COUNT(*) FROM silver_stores)
UNION ALL SELECT 'fx',        (SELECT COUNT(*) FROM bronze_exchange_rates), (SELECT COUNT(*) FROM silver_exchange_rates)

### Test 3: Spot-check the transformations
Expected: Miloš with a proper š; `Wandearah East` (was WANDEARAH EAST); price 12.99 as a number; subcategory `0101`.

In [ ]:
%%sql
SELECT customer_key, customer_name, city, birth_date FROM silver_customers WHERE customer_key = 301 OR customer_name LIKE 'Milo%'

In [ ]:
%%sql
SELECT product_key, product_name, unit_cost_usd, unit_price_usd, subcategory_key FROM silver_products WHERE product_key IN (1, 2)

### Test 4: The delivery business rule
Expected: Online = 13,165 rows, all with a delivery date (1–17 days). In-store = 49,719 rows, none with a delivery date.

In [ ]:
%%sql
SELECT CASE WHEN is_online THEN 'Online' ELSE 'In-store' END AS channel,
       COUNT(*) AS order_lines, COUNT(delivery_date) AS with_delivery_date,
       MIN(delivery_days) AS min_days, ROUND(AVG(delivery_days), 2) AS avg_days, MAX(delivery_days) AS max_days
FROM silver_sales GROUP BY is_online